# First Importations

In [3]:
# First we import all the necessary libraries I think I will use
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import re
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

In [4]:
# Now the first thing we need to do is to define the sentiment as it is defined in the exercise.

sentiments = {
    "LABEL_0": "Bearish",
    "LABEL_1": "Bullish",
    "LABEL_2": "Neutral"
}

# Data loading and preprocessing

In [ ]:
# We load the training and validation datasets from CSV files
train_df = pd.read_csv("sent_train.csv")
valid_df = pd.read_csv("sent_valid.csv")

print(f"Training samples:   {len(train_df)}")
print(f"Validation samples: {len(valid_df)}")
print(f"\nLabel distribution (train):\n{train_df['label'].value_counts()}")
print(f"\nLabel distribution (valid):\n{valid_df['label'].value_counts()}")

def preprocess_text(text):

    # We need to clean each tweet by removing URLs, special characters and converting everything to lowercase.

    # We remove URLs because links are common in financial tweets)
    text = re.sub(r"http\S+|www\S+", "", text)
    # We remove stock tickers like $AAPL, $BYND 
    text = re.sub(r"\$[A-Z]+", "", text)
    # We remove special characters but keep alphanumeric and spaces
    text = re.sub(r"[^a-zA-Z0-9\s]", "", text)
    # We convert to lowercase and strip extra whitespace
    text = text.lower().strip()
    return text


# We apply the preprocessing function to both datasets
train_df["clean_text"] = train_df["text"].apply(preprocess_text)
valid_df["clean_text"] = valid_df["text"].apply(preprocess_text)

# We can now look at some examples of the cleaned text to verify our preprocessing
print("\nSample cleaned tweets:")
for i in range(2):
    print(f"Original: {train_df['text'][i]}")
    print(f"Cleaned:  {train_df['clean_text'][i]}\n")

Training samples:   9543
Validation samples: 2388

Label distribution (train):
label
2    6178
1    1923
0    1442
Name: count, dtype: int64

Label distribution (valid):
label
2    1566
1     475
0     347
Name: count, dtype: int64

Sample cleaned tweets:
Original: $BYND - JPMorgan reels in expectations on Beyond Meat https://t.co/bd0xbFGjkT
Cleaned:  jpmorgan reels in expectations on beyond meat

Original: $CCL $RCL - Nomura points to bookings weakness at Carnival and Royal Caribbean https://t.co/yGjpT2ReD3
Cleaned:  nomura points to bookings weakness at carnival and royal caribbean



# Vocabulary Building

In [ ]:
# We tokenize by splitting on whitespace
def tokenize(text):
    return text.split()

# We build the vocabulary from the training set only, we learn this last year, if we use the validation set to build the vocab, we are leaking information 
# from the validation set into the training process, which can lead to overfitting.
all_tokens = []
for text in train_df["clean_text"]:
    all_tokens.extend(tokenize(text))

# We count token frequencies to filter rare words
token_counts = Counter(all_tokens)

# We only keep tokens that appear at least 2 times to reduce noise
MIN_FREQ = 2
vocab = ["<PAD>", "<UNK>"] + [
    token for token, count in token_counts.items() if count >= MIN_FREQ
]